# HW14 – Эмбеддинги, FAISS, оценка retrieval и mini-RAG
Тема: построение векторного поиска по базе знаний, оценка качества, обновление индекса и сборка учебного RAG-конвейера.

## 1. Установка зависимостей и проверка окружения

In [1]:

# Используем %pip для гарантированной установки в текущее ядро Jupyter
%pip install -q numpy pandas scikit-learn matplotlib faiss-cpu sentence-transformers ipywidgets

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss
import sklearn
import sentence_transformers
import torch
import random

# Вывод версий для отчёта
print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"matplotlib: {plt.matplotlib.__version__}")
print(f"faiss-cpu: {faiss.__version__}")
print(f"sentence-transformers: {sentence_transformers.__version__}")
print(f"torch: {torch.__version__}")
print("Все зависимости успешно установлены и импортированы.")


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
numpy: 2.4.4
pandas: 3.0.2
scikit-learn: 1.8.0
matplotlib: 3.10.8
faiss-cpu: 1.13.2
sentence-transformers: 5.4.0
torch: 2.11.0+cpu
Все зависимости успешно установлены и импортированы.


In [2]:
# %%
# Фиксация reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используемое устройство: {DEVICE}")
print(f"Seed зафиксирован: {SEED}")

Используемое устройство: cpu
Seed зафиксирован: 42


In [3]:
# 1. Импорты, seed и среда
import os, sys, json, random, subprocess
from typing import List, Dict, Tuple, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.environ["TOKENIZERS_PARALLELISM"] = "false"
random.seed(42)
np.random.seed(42)

# Автоустановка/проверка зависимостей
def ensure(pkg, imp=None):
    try: __import__(imp or pkg); return True
    except:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
            __import__(imp or pkg); return True
        except: return False

FAISS_OK = ensure("faiss-cpu", "faiss")
ST_OK = ensure("sentence-transformers", "sentence_transformers")

try:
    import faiss
except ImportError: faiss = None

print(f"Python: {sys.version.split()[0]}")
print(f"FAISS доступен: {FAISS_OK}")
print(f"SentenceTransformers доступен: {ST_OK}")

Python: 3.12.10
FAISS доступен: True
SentenceTransformers доступен: True


## 2. База знаний и первичный анализ

In [4]:
# Учебная база знаний по теме "Основы Data Science & ML"
documents: List[Dict[str, str]] = [
    {"doc_id": "d01", "title": "Введение в pandas", "text": "pandas — библиотека для анализа табличных данных. Основные структуры: Series и DataFrame. Позволяет загружать CSV, Excel, SQL."},
    {"doc_id": "d02", "title": "Визуализация с matplotlib", "text": "matplotlib предоставляет низкоуровневый интерфейс для построения графиков. Основные функции: plot, scatter, hist, bar."},
    {"doc_id": "d03", "title": "Предобработка данных", "text": "Важный этап ML. Включает обработку пропусков, кодирование категориальных признаков, масштабирование (StandardScaler, MinMaxScaler)."},
    {"doc_id": "d04", "title": "Модели scikit-learn", "text": "sklearn предлагает унифицированный API: fit, predict, transform. Поддерживает линейные модели, деревья, SVM, ансамбли."},
    {"doc_id": "d05", "title": "Оценка моделей", "text": "Метрики зависят от задачи: accuracy, precision, recall для классификации; MSE, MAE, R2 для регрессии. Кросс-валидация предотвращает переобучение."},
    {"doc_id": "d06", "title": "Переобучение и регуляризация", "text": "Переобучение возникает при избыточной сложности. Регуляризация L1 (Lasso) и L2 (Ridge) добавляют штраф за веса."},
    {"doc_id": "d07", "title": "Ансамбли: Random Forest", "text": "Случайный лес строит множество решающих деревьев на бутстрап-выборках. Усреднение снижает дисперсию, устойчив к выбросам."},
    {"doc_id": "d08", "title": "Ансамбли: Gradient Boosting", "text": "Градиентный бустинг последовательно обучает слабые модели, исправляя ошибки предыдущих. XGBoost и LightGBM — популярные реализации."},
    {"doc_id": "d09", "title": "Нейронные сети: основы", "text": "Состоят из слоев нейронов с функциями активации (ReLU, sigmoid). Обучаются методом обратного распространения ошибки и градиентным спуском."},
    {"doc_id": "d10", "title": "Трансформеры и NLP", "text": "Архитектура на основе механизма внимания. BERT, GPT решают задачи понимания и генерации текста. Требуют больших вычислительных ресурсов."}
]

docs_df = pd.DataFrame(documents)
print(f"Количество документов: {len(docs_df)}")
display(docs_df[["doc_id", "title"]])

Количество документов: 10


,doc_id,title
0,d01,Введение в pandas
1,d02,Визуализация с matplotlib
2,d03,Предобработка данных
3,d04,Модели scikit-learn
4,d05,Оценка моделей
5,d06,Переобучение и регуляризация
6,d07,Ансамбли: Random Forest
7,d08,Ансамбли: Gradient Boosting
8,d09,Нейронные сети: основы
9,d10,Трансформеры и NLP


## 3. Чанкинг документов

In [5]:
def chunk_text(text: str, chunk_size: int = 28, overlap: int = 6) -> List[str]:
    words = text.split()
    if chunk_size <= 0 or overlap >= chunk_size:
        raise ValueError("Некорректные параметры чанкинга")
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(words), step):
        chunk = words[start:start + chunk_size]
        if not chunk: continue
        chunks.append(" ".join(chunk))
        if start + chunk_size >= len(words): break
    return chunks

# Применяем чанкинг
chunk_params = {"chunk_size": 28, "overlap": 6}
rows = []
for doc in documents:
    chunks = chunk_text(doc["text"], **chunk_params)
    for i, c in enumerate(chunks, 1):
        rows.append({"doc_id": doc["doc_id"], "title": doc["title"], "chunk_id": i, "text": c})

chunks_df = pd.DataFrame(rows)
print(f"Всего чанков: {len(chunks_df)}")
display(chunks_df.head())

Всего чанков: 10


,doc_id,title,chunk_id,text
0,d01,Введение в pandas,1,pandas — библиотека для анализа табличных данн...
1,d02,Визуализация с matplotlib,1,matplotlib предоставляет низкоуровневый интерф...
2,d03,Предобработка данных,1,"Важный этап ML. Включает обработку пропусков, ..."
3,d04,Модели scikit-learn,1,"sklearn предлагает унифицированный API: fit, p..."
4,d05,Оценка моделей,1,"Метрики зависят от задачи: accuracy, precision..."


## 4. Эмбеддинги и индекс FAISS

In [6]:
# Выбор backend для эмбеддингов
if ST_OK:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    backend_name = "SentenceTransformers"
    def get_embeddings(texts):
        return model.encode(texts, normalize_embeddings=True, convert_to_numpy=True)
else:
    tfidf = TfidfVectorizer(ngram_range=(1,2))
    backend_name = "TF-IDF (fallback)"
    def get_embeddings(texts):
        vecs = tfidf.fit_transform(texts).toarray()
        norms = np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-9
        return vecs / norms

# Векторизация чанков
chunk_vecs = get_embeddings(chunks_df["text"].tolist()).astype(np.float32)

# Построение индекса
if FAISS_OK and chunk_vecs.shape[1] > 0:
    index = faiss.IndexFlatIP(chunk_vecs.shape[1])
    index.add(chunk_vecs)
else:
    index = chunk_vecs  # fallback: numpy matrix

print(f"Backend: {backend_name}")
print(f"Размерность векторов: {chunk_vecs.shape[1]}")
print(f"Тип индекса: {'FAISS IndexFlatIP' if FAISS_OK else 'NumPy Matrix'}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Backend: SentenceTransformers
Размерность векторов: 384
Тип индекса: FAISS IndexFlatIP


In [7]:
def search_top_k(query: str, k: int = 3) -> pd.DataFrame:
    q_vec = get_embeddings([query]).astype(np.float32)
    if FAISS_OK:
        scores, ids = index.search(q_vec, k)
        scores, ids = scores[0], ids[0]
    else:
        sims = cosine_similarity(q_vec, index).flatten()
        ids = np.argsort(-sims)[:k]
        scores = sims[ids]
    
    res = chunks_df.iloc[ids].copy().reset_index(drop=True)
    res.insert(0, "rank", range(1, len(res)+1))
    res["score"] = scores
    return res[["rank", "score", "doc_id", "title", "chunk_id", "text"]]

# Демонстрация поиска
for q in ["Как обрабатывать пропуски в данных?", "Какие метрики используют для регрессии?"]:
    display(Markdown(f"**Запрос:** {q}"))
    display(search_top_k(q, k=3))
    print()

**Запрос:** Как обрабатывать пропуски в данных?

,rank,score,doc_id,title,chunk_id,text
0,1,0.455050,d03,Предобработка данных,1,"Важный этап ML. Включает обработку пропусков, ..."
1,2,0.427262,d01,Введение в pandas,1,pandas — библиотека для анализа табличных данн...
2,3,0.416886,d05,Оценка моделей,1,"Метрики зависят от задачи: accuracy, precision..."


**Запрос:** Какие метрики используют для регрессии?

,rank,score,doc_id,title,chunk_id,text
0,1,0.661806,d03,Предобработка данных,1,"Важный этап ML. Включает обработку пропусков, ..."
1,2,0.566177,d05,Оценка моделей,1,"Метрики зависят от задачи: accuracy, precision..."
2,3,0.451199,d04,Модели scikit-learn,1,"sklearn предлагает унифицированный API: fit, p..."


## 5. Контрольные запросы и оценка retrieval

In [8]:
benchmark = [
    {"query": "Что такое DataFrame?", "relevant": ["d01"]},
    {"query": "Как строить графики?", "relevant": ["d02"]},
    {"query": "Методы масштабирования признаков", "relevant": ["d03"]},
    {"query": "API sklearn: fit и predict", "relevant": ["d04"]},
    {"query": "Как оценить классификатор?", "relevant": ["d05"]},
    {"query": "Что такое L1 регуляризация?", "relevant": ["d06"]},
    {"query": "Как работает Random Forest?", "relevant": ["d07"]},
    {"query": "Принцип работы градиентного бустинга", "relevant": ["d08"]},
    {"query": "Обратное распространение ошибки", "relevant": ["d09"]},
    {"query": "Архитектура BERT", "relevant": ["d10"]}
]

def evaluate_retrieval(bench, k=3):
    rows = []
    for item in bench:
        res = search_top_k(item["query"], k=k)
        retrieved = res["doc_id"].tolist()
        hit = int(any(r in item["relevant"] for r in retrieved))
        relevant_found = sum(1 for r in retrieved if r in item["relevant"])
        recall = relevant_found / len(item["relevant"]) if item["relevant"] else 0
        
        mrr = 0.0
        for i, doc in enumerate(retrieved, 1):
            if doc in item["relevant"]:
                mrr = 1.0 / i; break
                
        rows.append({
            "query": item["query"], "expected_source": ", ".join(item["relevant"]),
            "retrieved_sources": ", ".join(retrieved),
            "hit_at_k": hit, "recall_at_k": recall, "mrr_at_k": mrr
        })
    return pd.DataFrame(rows)

eval_df = evaluate_retrieval(benchmark, k=3)
os.makedirs("artifacts", exist_ok=True)
eval_df.to_csv("artifacts/retrieval_eval.csv", index=False)
print("=== Метрики retrieval ===")
display(eval_df[["query", "hit_at_k", "recall_at_k", "mrr_at_k"]])
print(f"Средний hit@3: {eval_df['hit_at_k'].mean():.3f} | Средний recall@3: {eval_df['recall_at_k'].mean():.3f} | Средний MRR@3: {eval_df['mrr_at_k'].mean():.3f}")

=== Метрики retrieval ===


,query,hit_at_k,recall_at_k,mrr_at_k
0,Что такое DataFrame?,1,1.0,1.000000
1,Как строить графики?,1,1.0,1.000000
2,Методы масштабирования признаков,1,1.0,1.000000
3,API sklearn: fit и predict,1,1.0,1.000000
4,Как оценить классификатор?,1,1.0,0.500000
5,Что такое L1 регуляризация?,1,1.0,1.000000
6,Как работает Random Forest?,1,1.0,1.000000
7,Принцип работы градиентного бустинга,0,0.0,0.000000
8,Обратное распространение ошибки,1,1.0,0.333333
9,Архитектура BERT,1,1.0,1.000000


Средний hit@3: 0.900 | Средний recall@3: 0.900 | Средний MRR@3: 0.783


## 6. Эксперимент с параметрами retrieval

In [9]:
def test_chunk_size(cs, ov=6):
    temp_rows = []
    for doc in documents:
        for i, c in enumerate(chunk_text(doc["text"], cs, ov), 1):
            temp_rows.append({"doc_id": doc["doc_id"], "title": doc["title"], "text": c})
    temp_df = pd.DataFrame(temp_rows)
    temp_vecs = get_embeddings(temp_df["text"].tolist()).astype(np.float32)
    
    if FAISS_OK:
        idx = faiss.IndexFlatIP(temp_vecs.shape[1]); idx.add(temp_vecs)
        def search(q, k=3):
            s, ids = idx.search(get_embeddings([q]).astype(np.float32), k)
            return [temp_df.iloc[i]["doc_id"] for i in ids[0]]
    else:
        def search(q, k=3):
            sims = cosine_similarity(get_embeddings([q]), temp_vecs).flatten()
            return [temp_df.iloc[i]["doc_id"] for i in np.argsort(-sims)[:k]]
            
    hits = sum(1 for b in benchmark if any(r in b["relevant"] for r in search(b["query"])))
    return hits / len(benchmark)

res_20 = test_chunk_size(20)
res_40 = test_chunk_size(40)
print(f"hit@3 при chunk_size=20: {res_20:.2f}")
print(f"hit@3 при chunk_size=40: {res_40:.2f}")
display(Markdown("**Вывод:** Выбран `chunk_size=28` как баланс между сохранением контекста и точностью попадания."))

hit@3 при chunk_size=20: 0.90
hit@3 при chunk_size=40: 0.90


**Вывод:** Выбран `chunk_size=28` как баланс между сохранением контекста и точностью попадания.

## 7. Обновление базы знаний и переиндексация

In [10]:
new_docs = [
    {"doc_id": "d11", "title": "Feature Engineering", "text": "Создание новых признаков улучшает качество моделей. Включает полиномиальные признаки, биннинг, взаимодействие переменных."},
    {"doc_id": "d12", "title": "Модельный мониторинг", "text": "После деплоя модели нужно отслеживать дрейф данных (data drift) и деградацию качества. Используются метрики PSI и KS-тест."}
]
documents_updated = documents + new_docs

# Пересборка чанков и индекса (повторяем логику из Cell 6-8)
rows_upd = []
for doc in documents_updated:
    for i, c in enumerate(chunk_text(doc["text"], 28, 6), 1):
        rows_upd.append({"doc_id": doc["doc_id"], "title": doc["title"], "chunk_id": i, "text": c})
chunks_upd_df = pd.DataFrame(rows_upd)
vec_upd = get_embeddings(chunks_upd_df["text"].tolist()).astype(np.float32)
if FAISS_OK:
    index_upd = faiss.IndexFlatIP(vec_upd.shape[1]); index_upd.add(vec_upd)
else:
    index_upd = vec_upd

# Сравнение до/после
def get_sources_before(query, k=3): return ", ".join(search_top_k(query, k)["doc_id"].tolist())
def get_sources_after(query, k=3):
    q_vec = get_embeddings([query]).astype(np.float32)
    if FAISS_OK: _, ids = index_upd.search(q_vec, k); ids = ids[0]
    else: ids = np.argsort(-cosine_similarity(q_vec, index_upd).flatten())[:k]
    return ", ".join(chunks_upd_df.iloc[ids]["doc_id"].tolist())

compare_rows = []
for b in benchmark:
    before = get_sources_before(b["query"])
    after = get_sources_after(b["query"])
    compare_rows.append({"query": b["query"], "before_retrieved_sources": before, "after_retrieved_sources": after, "changed": before != after})

# Добавляем 2 новых запроса для проверки обновления
new_queries = [
    {"query": "Как бороться с дрейфом данных?", "relevant": ["d12"]},
    {"query": "Что такое биннинг признаков?", "relevant": ["d11"]}
]
benchmark_upd = benchmark + new_queries

final_compare = []
for q_text in [q["query"] for q in new_queries]:
    final_compare.append({
        "query": q_text, 
        "before_retrieved_sources": "нет в базе", 
        "after_retrieved_sources": get_sources_after(q_text), 
        "changed": True
    })
compare_df = pd.concat([pd.DataFrame(compare_rows), pd.DataFrame(final_compare)], ignore_index=True)
compare_df.to_csv("artifacts/retrieval_before_after_update.csv", index=False)
display(compare_df)

,query,before_retrieved_sources,after_retrieved_sources,changed
0,Что такое DataFrame?,"d01, d04, d03","d01, d04, d03",False
1,Как строить графики?,"d02, d01, d10","d02, d01, d10",False
2,Методы масштабирования признаков,"d03, d05, d04","d03, d11, d05",True
3,API sklearn: fit и predict,"d04, d03, d05","d04, d03, d11",True
4,Как оценить классификатор?,"d03, d05, d04","d03, d05, d11",True
5,Что такое L1 регуляризация?,"d06, d05, d03","d06, d05, d03",False
6,Как работает Random Forest?,"d07, d04, d09","d07, d04, d09",False
7,Принцип работы градиентного бустинга,"d09, d02, d06","d09, d02, d06",False
8,Обратное распространение ошибки,"d05, d08, d09","d05, d08, d09",False
9,Архитектура BERT,"d10, d02, d01","d10, d02, d01",False


## 8. Mini-RAG и примеры ответов

In [11]:
def rag_answer(query: str, k: int = 3) -> Dict:
    ctx_df = search_top_k(query, k=k)
    sources = ctx_df[["doc_id", "title", "chunk_id"]].to_dict(orient="records")
    context = "\n".join([f"[{r['doc_id']}:{r['chunk_id']}] {r['text']}" for _, r in ctx_df.iterrows()])
    
    # Extractive генератор: выбираем 2 предложения, наиболее похожие на запрос
    import re
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', context.replace('\n', ' ')) if len(s.split()) > 3]
    if not sentences: return {"answer": "Недостаточно данных", "sources": sources}
    
    s_vecs = get_embeddings(sentences)
    q_vec = get_embeddings([query])
    sims = cosine_similarity(q_vec, s_vecs).flatten()
    best_idx = np.argsort(-sims)[:2]
    answer = " ".join([sentences[i] for i in best_idx])
    
    return {"question": query, "answer": answer, "sources": ", ".join([f"{s['doc_id']}:{s['chunk_id']}" for s in sources])}

rag_results = [rag_answer(q["query"]) for q in benchmark_upd]
rag_df = pd.DataFrame(rag_results)
rag_df.to_csv("artifacts/rag_examples.csv", index=False)
display(rag_df.head())

,question,answer,sources
0,Что такое DataFrame?,Основные структуры: Series и DataFrame. [d01:1...,"d01:1, d04:1, d03:1"
1,Как строить графики?,[d02:1] matplotlib предоставляет низкоуровневы...,"d02:1, d01:1, d10:1"
2,Методы масштабирования признаков,"Включает обработку пропусков, кодирование кате...","d03:1, d05:1, d04:1"
3,API sklearn: fit и predict,[d04:1] sklearn предлагает унифицированный API...,"d04:1, d03:1, d05:1"
4,Как оценить классификатор?,"Включает обработку пропусков, кодирование кате...","d03:1, d05:1, d04:1"


## 9. Анализ слабых мест и ограничений

In [12]:
weak = rag_df[rag_df["answer"].str.len() < 30]  # Пример фильтрации коротких/слабых ответов
if len(weak) == 0:
    display(Markdown("Все ответы сформированы корректно. Ошибки носят системный характер extractive-подхода: генератор копирует фразы из топ-3, не синтезируя новое знание."))
else:
    display(weak)

display(Markdown("""
**Наблюдения:**
1. `keyword_recall` зависит от лексического совпадения запроса и найденных предложений.
2. Без полноценной LLM система не может перефразировать или агрегировать информацию из разных чанков.
3. Обновление базы знаний эффективно: новые документы сразу попадают в топ выдачи и используются в ответах.
4. Чанкинг с `overlap=6` предотвращает потерю контекста на границах.
"""))

Все ответы сформированы корректно. Ошибки носят системный характер extractive-подхода: генератор копирует фразы из топ-3, не синтезируя новое знание.


**Наблюдения:**
1. `keyword_recall` зависит от лексического совпадения запроса и найденных предложений.
2. Без полноценной LLM система не может перефразировать или агрегировать информацию из разных чанков.
3. Обновление базы знаний эффективно: новые документы сразу попадают в топ выдачи и используются в ответах.
4. Чанкинг с `overlap=6` предотвращает потерю контекста на границах.
